# Batch the phenotype list into FelixGenome run-table rows

A genome-wide scan over the whole phenotype list is `71P + 92` Cromwell calls in
one workflow. This splits the list across submissions: each batch is a
`flare_lai_prod_set` row differing from the template only in
`selected_phenotypes`, so the GRM and all 22 chromosome packs call-cache across
batches and the saving costs nothing.

Run [`tractor_01_prepare_inputs.ipynb`](tractor_01_prepare_inputs.ipynb) first —
it writes the phenotype list and `pheno_cov.tsv` this notebook reads.

Knobs, all environment variables:

| Variable | Default | What it does |
| --- | --- | --- |
| `WGS_PREFIX` | `felix/phenotypes/wgs` | where the prepare notebook put the list |
| `FELIX_TEMPLATE_ROW` | `felix.wgs.full.v1` | run-table row to clone. Full covariates are the production arm; `felix.wgs.limited.v1` is the pilot's comparator and is not run going forward |
| `FELIX_BATCH_SIZE` | `95` | phenotypes per submission, about 6,800 calls |
| `FELIX_ROW_PREFIX` | `felix.wgs<N>.full` | id prefix for the generated rows |
| `FELIX_BATCH_PREFIX` | `felix/phenotypes/wgs<N>` | bucket path for the per-batch lists |
| `FELIX_RUN_TABLE_TSV` | fetched from the bucket | local template to clone instead |

Details and the upload order: `felix/configs/terra/RUN_TABLE.md`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Bootstrap scripts/ from $WORKSPACE_BUCKET/scripts/ when not on the VM.
for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError(
            "scripts/ not found locally and WORKSPACE_BUCKET is unset. "
            "Upload scripts/ to gs://WORKSPACE/scripts/."
        )
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(
        ["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"]
    )
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook
SCRIPTS = init_notebook("make_felix_run_table.py")

make_table = SCRIPTS / "make_felix_run_table.py"
assert make_table.is_file(), make_table
WS = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
assert WS, "WORKSPACE_BUCKET is unset"
print("script :", make_table)
print("bucket :", WS)


In [ ]:
def sh(cmd: str, check: bool = True) -> int:
    """Run a shell command and actually fail on a non-zero exit.

    `get_ipython().system()` returns None even when the command fails, so the
    usual `if rc: raise` never fires and the notebook sails past a failed
    gsutil into a confusing FileNotFoundError further down.
    """
    print(cmd)
    rc = subprocess.run(cmd, shell=True).returncode
    if check and rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc


# The phenotype list produced by tractor_01_prepare_inputs. Its order is the
# contract: null arrays are indexed against it, so the batches must preserve it.
PHENO_PREFIX = os.environ.get("WGS_PREFIX", "felix/phenotypes/wgs").strip("/")
PHENOTYPES = Path("selected_phenotypes.txt")
if not PHENOTYPES.exists():
    sh(f"gsutil cp {WS}/{PHENO_PREFIX}/selected_phenotypes.txt {PHENOTYPES}")

PHENO_COV = f"{WS}/{PHENO_PREFIX}/pheno_cov.tsv"

# The run-table row to clone. Try, in order: an explicit local copy, the bucket
# (put there by stage_felix_scripts.sh), then a repo clone on this VM.
TEMPLATE = Path(os.environ.get("FELIX_RUN_TABLE_TSV", "felix_runs.entity.tsv"))
if not TEMPLATE.exists():
    if sh(f"gsutil cp {WS}/felix/configs/felix_runs.entity.tsv {TEMPLATE}", check=False):
        rel = Path("felix/configs/terra/felix_runs.entity.tsv")
        local = next(
            (c / rel for c in (Path.cwd(), *Path.cwd().parents, Path.home())
             if (c / rel).is_file()),
            None,
        )
        if local is None:
            raise FileNotFoundError(
                f"No run-table template. {WS}/felix/configs/felix_runs.entity.tsv "
                "does not exist, so stage_felix_scripts.sh has not been run since "
                "it started staging configs:\n"
                "    bash scripts/stage_felix_scripts.sh\n"
                "Sync the repo first if that script has no 'run-table configuration' "
                "section. Or point FELIX_RUN_TABLE_TSV at a local copy."
            )
        print(f"bucket copy absent; using the repo clone at {local}")
        TEMPLATE.write_text(local.read_text())

# Full covariates are the production arm. The limited set was the pilot's
# comparator; it has served its purpose and is not run going forward.
TEMPLATE_ROW = os.environ.get("FELIX_TEMPLATE_ROW", "felix.wgs.full.v1")
ARM = "limited" if "limited" in TEMPLATE_ROW else "full"
COVARIATE_FILE = f"covariate_columns_{ARM}.txt"
BATCH_SIZE = int(os.environ.get("FELIX_BATCH_SIZE", "95"))
OUT_DIR = Path(os.environ.get("FELIX_RUN_TABLE_OUT", "felix_run_table"))

n_phenos = len([ln for ln in PHENOTYPES.read_text().split() if ln])
ROW_PREFIX = os.environ.get("FELIX_ROW_PREFIX", f"felix.wgs{n_phenos}.{ARM}")
BATCH_PREFIX = os.environ.get("FELIX_BATCH_PREFIX", f"felix/phenotypes/wgs{n_phenos}")

rows = {r.split("\t")[0] for r in TEMPLATE.read_text().splitlines()[1:]}
assert TEMPLATE_ROW in rows, f"{TEMPLATE_ROW} not in {sorted(rows)}"

print(f"{n_phenos:,} phenotypes -> {-(-n_phenos // BATCH_SIZE)} batches of {BATCH_SIZE}")
print(f"  template row : {TEMPLATE_ROW}")
print(f"  row prefix   : {ROW_PREFIX}")
print(f"  pheno_cov    : {PHENO_COV}")
print(f"  calls        : {71 * n_phenos + 92:,} as one workflow, "
      f"~{71 * BATCH_SIZE + 92:,} per batch")


In [ ]:
sh(
    f"python3 {make_table} "
    f"--phenotypes {PHENOTYPES} "
    f"--template {TEMPLATE} "
    f"--template-row {TEMPLATE_ROW} "
    f"--row-prefix {ROW_PREFIX} "
    f"--batch-size {BATCH_SIZE} "
    f"--bucket {WS} "
    f"--phenotype-prefix {BATCH_PREFIX} "
    f"--pheno-cov {PHENO_COV} "
    # the cohort files the full-list selection wrote, so a row points at one
    # prefix rather than mixing in the pilot's
    f"--set analysis_samples={WS}/{PHENO_PREFIX}/analysis_samples.txt "
    f"--set covariate_columns={WS}/{PHENO_PREFIX}/{COVARIATE_FILE} "
    f"--out-dir {OUT_DIR}"
)

import pandas as pd

entity = pd.read_csv(OUT_DIR / "felix_runs.entity.tsv", sep="\t", dtype=str)
member = pd.read_csv(OUT_DIR / "felix_runs.membership.tsv", sep="\t", dtype=str)
batch_files = sorted((OUT_DIR / "phenotypes").glob("*.txt"))

# Every phenotype appears once, in order, across the batch files.
seen = [p for f in batch_files for p in f.read_text().split()]
assert seen == PHENOTYPES.read_text().split(), "batches do not reproduce the phenotype list"
assert len(entity) == len(batch_files) == member.iloc[:, 0].nunique()
assert len(member) == len(entity) * 22, "every row needs all 22 autosomes"
print(f"{len(entity)} rows, {len(member)} membership rows, "
      f"{len(seen):,} phenotypes covered exactly once")
display(entity[["entity:flare_lai_prod_set_id", "selected_phenotypes",
                "pheno_cov", "covariate_columns", "min_mac"]])


In [ ]:
# Copy the per-batch phenotype files to the bucket the table cells point at.
sh(f"gsutil -m cp {OUT_DIR}/phenotypes/*.txt {WS}/{BATCH_PREFIX}/")
sh(f"gsutil ls {WS}/{BATCH_PREFIX}/ | head -5")

# Confirm every selected_phenotypes cell resolves to an object that now exists.
present = set(subprocess.run(
    ["gsutil", "ls", f"{WS}/{BATCH_PREFIX}/"], capture_output=True, text=True
).stdout.split())
missing = [u for u in entity["selected_phenotypes"] if u not in present]
assert not missing, missing
print(f"all {len(entity)} selected_phenotypes cells resolve")

print("\nUpload to Terra (Data tab), membership first, then the entity TSV:")
print(f"  {OUT_DIR / 'felix_runs.membership.tsv'}")
print(f"  {OUT_DIR / 'felix_runs.entity.tsv'}")
print("\nThen submit the FelixGenome method config on one row and let it finish;")
print("the GRM, the 22 packs and the repeat aggregation cache for the rest.")
